In [ ]:
# Scrape population data from Wikipedia, clean string numbers to integers and add current date.

import re
from datetime import date
import pandas as pd
import requests
from bs4 import BeautifulSoup


def get_population_data(city_list):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }

    population_data = []
    today_date = date.today().strftime("%Y-%m-%d")

    for city in city_list:
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            soup = BeautifulSoup(response.content, "html.parser")
            infobox = soup.find("table", {"class": "infobox"})

            population_val = None

            if infobox:
                # Iterate through table rows to locate Population label
                for row in infobox.find_all("tr"):
                    row_text = row.get_text()
                    if "Population" in row_text:
                        next_row = row.find_next_sibling("tr")
                        if next_row:
                            # Extract text and remove non-numeric characters (keep only digits)
                            raw_pop = next_row.get_text()
                            clean_pop = re.sub(r"[^\d]", "", raw_pop)
                            if clean_pop:
                                population_val = int(clean_pop)
                        break

            population_data.append(
                {
                    "city_name": city,
                    "population": population_val,
                    "retrieved_at": today_date,
                }
            )
            print(f"Population data fetched successfully for: {city}")
        else:
            print(f"Failed to fetch page for: {city}")

    return pd.DataFrame(population_data)


# Define target cities
cities = ["Berlin", "Hamburg", "Munich"]

# Fetch population data
df_population_raw = get_population_data(cities)
print(df_population_raw)

Population data fetched successfully for: Berlin
Population data fetched successfully for: Hamburg
Population data fetched successfully for: Munich
  city_name  population retrieved_at
0    Berlin     3596999   2026-09-28
1   Hamburg     1973896   2026-09-28
2    Munich     1505036   2026-09-28


In [ ]:
# Retrieve city_id mapping from MySQL, merge with population data
# insert into the 'population' table.

import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Load environment variables from .env file
load_dotenv()

# Retrieve credentials securely
user = "root"
password = os.getenv("MYSQL_PASSWORD")
host = "localhost"
port = "3306"
database = "Gans"

# Create connection engine
connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

# Fetch existing cities from MySQL to get their city_id values
df_cities_from_db = pd.read_sql("SELECT city_id, city_name FROM cities", con=engine)

# Merge raw population DataFrame with database city_id mapping
df_population_ready = pd.merge(df_population_raw, df_cities_from_db, on="city_name")

# Select only the columns that match the MySQL 'population' table schema
df_population_ready = df_population_ready[["city_id", "population", "retrieved_at"]]

# Insert the processed population records into MySQL 'population' table
df_population_ready.to_sql(name="population", con=engine, if_exists="append", index=False)

print("Population data successfully inserted into Gans database!")

Population data successfully inserted into Gans database!
